In [17]:
import pickle
from torch.utils.data import Subset
from torch_geometric.datasets import BA2MotifDataset
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv,global_mean_pool
from torch_geometric.loader import DataLoader
import torch.optim as optim
from tqdm import tqdm

In [ ]:
dataset=BA2MotifDataset(root='./data/BA2Motif')
with open("./data/splits.pkl", "rb") as f:
    splits=pickle.load(f)

train_dataset=Subset(dataset,splits["train"])
val_dataset=Subset(dataset,splits["val"])


In [8]:
class GCNGraphClf(nn.Module):
    def __init__(self,num_node_featuers,num_classes,hidden_dim=64,mlp_hidden_dim=32):
        super(GCNGraphClf,self).__init__()
        self.conv1=GCNConv(num_node_featuers,hidden_dim)
        self.conv2=GCNConv(hidden_dim,hidden_dim)
        self.conv3=GCNConv(hidden_dim,hidden_dim)

        self.mlp=nn.Sequential(
            nn.Linear(hidden_dim,mlp_hidden_dim),
            nn.ReLU(),
            nn.Linear(mlp_hidden_dim,num_classes)
        )
    
    def forward(self,x,edge_index,batch):
        x=F.relu(self.conv1(x,edge_index))
        x=F.relu(self.conv2(x,edge_index))
        x=F.relu(self.conv3(x,edge_index))
        x=global_mean_pool(x,batch)
        x=self.mlp(x)
        return x
        

In [20]:
config={
    'hidden_dim':64,
    'mlp_hidden_dim':32,
    'leanrning_rate':0.001,
    'weight_decay':0.001,
    'epochs':100,
    'batch_size':32,
    'patience':20           # early stopping
}

In [10]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cpu


In [18]:
train_loader=DataLoader(train_dataset,batch_size=config['batch_size'],shuffle=True)
validation_loader=DataLoader(val_dataset,batch_size=config['batch_size'],shuffle=False)

In [14]:
model=GCNGraphClf(dataset.num_node_features,dataset.num_classes).to(device)
optimizer=torch.optim.Adam(model.parameters(),lr=config['leanrning_rate'],weight_decay=config['weight_decay'])
criterion=nn.CrossEntropyLoss()

In [16]:
def trian_one_epoch(model,loader,optimizer,criterion,device):
    model.train()
    total_loss=0
    for data in loader:
        data=data.to(device)
        optimizer.zero_grad()
        output=model(data.x,data.edge_index,data.batch)
        loss=criterion(output,data.y)
        loss.backward()
        optimizer.step()
        total_loss+=loss.item()
    return total_loss/len(loader.dataset)

def evaluate(model , loader,device):
    model.eval()
    correct=0
    total=0
    with torch.no_grad():
        for data in loader:
            data=data.to(device)
            output=model(data.x,data.edge_index,data.batch)
            pred=output.argmax(dim=1)
            correct+=(pred==data.y).sum().item()
            total+=len(data.y)
    return correct/total

In [ ]:
best_val_acc=0.0
patience_counter=0
best_model_state=None

with tqdm(total=config["epochs"],desc="Training epochs",unit="epoch") as pbar_epoch:
    for epoch in range(config["epochs"]):
        train_loss=trian_one_epoch(model,train_loader,optimizer,criterion,device)
        val_acc=evaluate(model,validation_loader,device)
        pbar_epoch.set_postfix({"train_loss":train_loss,"val_acc":val_acc})
        if val_acc>best_val_acc:
            best_val_acc=val_acc
            patience_counter=0
            best_model_state=model.state_dict().copy()
        else:
            patience_counter+=1
            if patience_counter>=config["patience"]:
                tqdm.write(f"Early stopping at epoch {epoch+1}")
                break

model.load_state_dict(best_model_state)
tqdm.write(f"Best validation accuracy: {best_val_acc:.4f}")

Training epochs:   0%|          | 0/100 [00:04<?, ?epoch/s, train_loss=0.0216, val_acc=0.39]

Early stopping at epoch 21


In [28]:
model.eval()
for param in model.parameters():
    param.requires_grad=False

torch.save({
    'model_state_dict':best_model_state,
    'config':config,
    'best_val_acc':best_val_acc,
},"./models/GCN/best_gcn_model.pt"
)
tqdm.write(" Model frozen and saved as 'best_gcn_model.pt'")

 Model frozen and saved as 'best_gcn_model.pt'
